In [ ]:
import os, sys
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
# Cria a conexão Spark

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")

In [ ]:
project_path    = os.getcwd()
data_path_root  = "C:\\Marco Conti\\Projetos\\Dados\\"

'c:\\Marco Conti\\Projetos\\mais_einstein\\Ondas_Calor'

In [ ]:
df_tempetatura_2023 = spark.read.parquet(f"{data_path_root}ERA5-temperaturas\\2023\\ERA5_temperatura.parquet")
df_tempetatura_2024 = spark.read.parquet(f"{data_path_root}ERA5-temperaturas\\2024\\ERA5_temperatura.parquet")
df_tempetatura_2025 = spark.read.parquet(f"{data_path_root}ERA5-temperaturas\\2025\\ERA5_temperatura.parquet")

df_tempetatura = df_tempetatura_2023.union(df_tempetatura_2024).union(df_tempetatura_2025)

df_tempetatura.printSchema()
df_tempetatura.limit(10).show()

Obtem as estatísticas de temperatura por Ano:
- Temperatura mínima
- Temperatura máxima
- Temperatura média
- Percentil 5%
- Percentil 90%

In [ ]:
# Criar as colunas min, max, med e percentis por ANO
# De acordo com documento Elaborador Sara Lopes de Moraes para o VERACIS

df_base = (
    df_tempetatura
        .withColumn("ano", F.year("data_medicao"))
        # .withColumn("mes", F.month("data_medicao"))
)

df_stats = (
    df_base
    .groupBy(
        "ano",
        # "mes",
        "latitude",
        "longitude"
    )
    .agg(
        F.min("valor").alias("temp_min_ano"),
        F.max("valor").alias("temp_max_ano"),
        F.avg("valor").alias("temp_media_ano"),
        F.expr("percentile_approx(valor, 0.05)").alias("percentil_05_ano"),
        F.expr("percentile_approx(valor, 0.95)").alias("percentil_95_ano")
    )
)

df_stats.printSchema()
df_stats.show(10, truncate=False)

In [ ]:
df_stats.count() # 50554 

In [ ]:
# df_base.count() # 27.728.869
# df_dia.count()

In [ ]:
# Anexar dados estatíscos e percentis ao dado diário
# Será utilizados para calcular as ondas de calor (periodos de dias consecutivos)
df_dia = (
    df_base.join(
        df_stats.select(
            "ano",
            # "mes",
            "latitude",
            "longitude",
            "temp_min_ano",
            "temp_max_ano",
            "temp_media_ano",
            "percentil_05_ano",
            "percentil_95_ano"
        ),
        [
            "ano",
            # "mes",
            "latitude",
            "longitude"
        ],
        how="left"
    )
)

print("Número de registros no DataFrame diário:", df_dia.count()) # 151662
df_dia.printSchema()
df_dia.show(10, truncate=False)

In [ ]:
# Classificar extremos
df_dia = (
    df_dia
    .withColumn(
        "extremo_alto",
        F.when(F.col("valor") > F.col("percentil_95_ano")
              ,(F.col("valor") - F.col("percentil_95_ano"))).otherwise(0))
    .withColumn(
        "extremo_baixo",
        F.when(F.col("valor") < F.col("percentil_05_ano")
              ,(F.col("percentil_05_ano") - F.col("valor"))).otherwise(0)
    )
)

df_dia.printSchema()
df_dia.filter("extremo_baixo != 0 or extremo_alto != 0").show(10, truncate=False)
# (df_dia
#     .select('data_medicao', 'latitude', 'longitude', 'percentil_05_ano', 'valor', 'percentil_95_ano'
#            ,'extremo_alto', 'extremo_baixo' )
#     .orderBy("latitude", "longitude", "data_medicao")
#     .show(1000, truncate=False))

In [ ]:
# Identifica os dias quentes:

# Considera-se dia quente quando a temperatura excede o percentil 95 do mês

df_flag = \
    df_dia.withColumn("flag_dia_quente"
                     ,F.when(F.col("valor") > F.col("percentil_95_ano"), 1).otherwise(0)
    )

# Mantém apenas os dias que atenderam ao critério
df_quentes = df_flag.filter(F.col("flag_dia_quente") == 1)


# Janela ordenada por data para cada ponto geográfico
janela_loc = Window.partitionBy("latitude", "longitude").orderBy("data_medicao")

# Ao subtrair a ordem do registro (rn) da data, dias consecutivos geram
# exatamente o mesmo identificador de grupo (grupo_id)
df_eventos = df_quentes \
    .withColumn("rn", F.row_number().over(janela_loc)) \
    .withColumn("grupo_id", F.expr("date_sub(data_medicao, CAST(rn AS INT))"))


In [ ]:
df_eventos.count() # 1.364.922

In [ ]:
# 1. Identificação dos Eventos e Validação da Duração (Global, sem quebra de mês/ano)
df_eventos_duracao = (
    df_eventos
    .groupBy("latitude", "longitude", "grupo_id")
    .agg(
        F.count("data_medicao").alias("duracao_total_onda"),
        F.min("data_medicao").alias("inicio_onda"),
        F.max("data_medicao").alias("fim_onda")
    )
    .filter(F.col("duracao_total_onda") > 2) # Filtra apenas eventos reais (> 2 dias)
)

# 2. Retornar os DIAS INDIVIDUAIS das ondas válidas mantendo os metadados da onda
df_dias_em_onda = (
    df_eventos
    .join(df_eventos_duracao, ["latitude", "longitude", "grupo_id"], "inner")
    .select(
        "latitude", 
        "longitude", 
        "data_medicao", 
        "ano", 
        # "mes", 
        "grupo_id", 
        "duracao_total_onda"
    )
)

#### Adicionar as métricas:
<pre>
- Número de ondas de calor (N-OdC)      : Total de eventos de ondas de calor registrados em um determinado ano.
- Frequência das ondas de calor (F-OdC) : Número total de dias que compõem as ondas de calor ao longo do ano.
- Duração das ondas de calor (D-OdC)    : Duração, em dias, do evento de onda de calor mais longo registrado no ano.
--> Criar:
- Amplitude das ondas de calor (A-OdC)  : Maior valor da temperatura média diária observado durante eventos de onda de calor no ano.
- Magnitude das ondas de calor (M-OdC)  : Média da temperatura média diária considerando todos os dias de ocorrência de ondas de calor no ano.
</pre>

In [ ]:
df_dias_em_onda.printSchema()

In [ ]:
df_metricas_mensal = (
    df_dias_em_onda
    .groupBy("latitude", "longitude", "ano", "mes")
    .agg(
        # Número de ondas distintas que passaram/ocorreram neste mês específico
        F.countDistinct("grupo_id").alias("numero_ondas_calor"),
        
        # Total exato de dias sob onda de calor DENTRO deste mês
        F.count("data_medicao").alias("frequencia_dias_onda_calor"),
        
        # Duração total dos eventos que afetaram este mês (máxima e média)
        F.max("duracao_total_onda").alias("duracao_maxima_onda"),
        F.round(F.avg("duracao_total_onda"), 2).alias("duracao_media_ondas")
    )
    .orderBy("ano", "mes", "latitude", "longitude")
)

df_metricas_mensal.show(10, truncate=False)

In [ ]:
df_metricas_anual = (
    df_dias_em_onda
    .groupBy("latitude", "longitude", "ano")
    .agg(
        # Número de ondas distintas que tiveram ao menos um dia neste ano
        F.countDistinct("grupo_id").alias("numero_ondas_calor"),
        
        # Total exato de dias do ano passados em onda de calor
        F.count("data_medicao").alias("frequencia_dias_onda_calor"),
        
        # Duração máxima e média dos eventos ocorridos no ano
        F.max("duracao_total_onda").alias("duracao_maxima_onda"),
        F.round(F.avg("duracao_total_onda"), 2).alias("duracao_media_ondas")
    )
    .orderBy("ano", "latitude", "longitude")
)

df_metricas_anual.filter("latitude = -34.0 and longitude = -74.0").show(10, truncate=False)

In [ ]:
df_metricas_anual.filter("latitude = -34.0 and longitude = -74.0").show(10, truncate=False)